# 📊 Exploratory Data Analysis - Household Income Prediction

**Dataset:** Regression Dataset for Household Income Analysis  
**Source:** [Kaggle - Stealth Technologies](https://www.kaggle.com/datasets/stealthtechnologies/regression-dataset-for-household-income-analysis)  
**Objective:** Phân tích khám phá dữ liệu (EDA) để hiểu các yếu tố ảnh hưởng đến thu nhập hộ gia đình.

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries imported successfully!')

In [ ]:
# Load dataset
df = pd.read_csv('../data/data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head(10)

## 2. Data Overview & Quality Check

In [ ]:
# Data types and info
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Memory Usage ===')
print(f'Total: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')
print(f'\nTotal missing: {missing.sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary for numerical columns
df.describe().round(2)

In [ ]:
# Identify column types
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}):')
for col in cat_cols:
    print(f'  - {col}: {df[col].nunique()} unique values')

print(f'\nNumerical columns ({len(num_cols)}):')
for col in num_cols:
    print(f'  - {col}: range [{df[col].min()}, {df[col].max()}]')

## 3. Target Variable Analysis (Income)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['Income'], bins=50, color='#6366f1', edgecolor='white', alpha=0.8)
axes[0].axvline(df['Income'].mean(), color='red', linestyle='--', label=f'Mean: ${df["Income"].mean():,.0f}')
axes[0].axvline(df['Income'].median(), color='green', linestyle='--', label=f'Median: ${df["Income"].median():,.0f}')
axes[0].set_title('Income Distribution')
axes[0].set_xlabel('Annual Income (USD)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Log-transformed histogram
log_income = np.log1p(df['Income'])
axes[1].hist(log_income, bins=50, color='#06b6d4', edgecolor='white', alpha=0.8)
axes[1].set_title('Log-Transformed Income Distribution')
axes[1].set_xlabel('log(Income + 1)')
axes[1].set_ylabel('Frequency')

# Box plot
axes[2].boxplot(df['Income'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#8b5cf6', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[2].set_title('Income Box Plot')
axes[2].set_ylabel('Annual Income (USD)')

plt.tight_layout()
plt.suptitle('Target Variable (Income) Analysis', y=1.02, fontsize=16, fontweight='bold')
plt.show()

print(f'Skewness (raw):  {df["Income"].skew():.4f}')
print(f'Skewness (log):  {log_income.skew():.4f}')
print(f'Kurtosis (raw):  {df["Income"].kurtosis():.4f}')
print(f'Kurtosis (log):  {log_income.kurtosis():.4f}')

In [ ]:
# Percentile analysis
percentiles = [10, 25, 50, 75, 90, 95, 99]
print('=== Income Percentiles ===')
for p in percentiles:
    val = df['Income'].quantile(p / 100)
    print(f'  P{p:2d}: ${val:>12,.0f}')

# IQR outlier detection
Q1, Q3 = df['Income'].quantile(0.25), df['Income'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Income'] < Q1 - 1.5 * IQR) | (df['Income'] > Q3 + 1.5 * IQR)]
print(f'\n=== Outlier Analysis (IQR) ===')
print(f'  Q1: ${Q1:,.0f}  |  Q3: ${Q3:,.0f}  |  IQR: ${IQR:,.0f}')
print(f'  Lower bound: ${Q1 - 1.5 * IQR:,.0f}')
print(f'  Upper bound: ${Q3 + 1.5 * IQR:,.0f}')
print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

## 4. Numerical Features Distribution

In [ ]:
num_features = [c for c in num_cols if c != 'Income']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b']

for i, col in enumerate(num_features):
    ax = axes[i // 2, i % 2]
    ax.hist(df[col], bins=30, color=colors[i], edgecolor='white', alpha=0.8)
    ax.axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7,
               label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.suptitle('Numerical Features Distribution', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 5. Categorical Features Distribution

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
palette = sns.color_palette('husl', 10)

for i, col in enumerate(cat_cols):
    ax = axes[i // 3, i % 3]
    counts = df[col].value_counts()
    bars = ax.barh(counts.index, counts.values, color=palette[i], alpha=0.8, edgecolor='white')
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_xlabel('Count')
    
    # Add value labels
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('Categorical Features Distribution', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 6. Income by Categorical Features

In [ ]:
# Income by key categorical features
key_cats = ['Education_Level', 'Occupation', 'Employment_Status', 'Location']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, col in enumerate(key_cats):
    ax = axes[i // 2, i % 2]
    order = df.groupby(col)['Income'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='Income', order=order, ax=ax,
                palette='husl', showfliers=False)
    ax.set_title(f'Income by {col}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Income (USD)')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Income Distribution by Key Categories', y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# Median income by categories (table)
print('=== Median Income by Category ===')
for col in key_cats:
    print(f'\n--- {col} ---')
    table = df.groupby(col)['Income'].agg(['median', 'mean', 'count'])
    table.columns = ['Median', 'Mean', 'Count']
    table = table.sort_values('Median', ascending=False)
    for idx, row in table.iterrows():
        print(f'  {idx:25s}  Median: ${row["Median"]:>12,.0f}  Mean: ${row["Mean"]:>12,.0f}  (n={int(row["Count"]):,})')

## 7. Correlation Analysis

In [ ]:
# Encode categorical for correlation analysis
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with Income (sorted)
income_corr = corr_matrix['Income'].drop('Income').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#10b981' if v > 0 else '#ef4444' for v in income_corr.values]
income_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white', alpha=0.8)
ax.set_title('Correlation with Income', fontweight='bold', fontsize=14)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(0, color='gray', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print('=== Correlation with Income ===')
for feat, corr in income_corr.items():
    print(f'  {feat:40s} {corr:+.4f}')

## 8. Scatter Plots: Numerical Features vs Income

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_scatter = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b']

for i, col in enumerate(num_features):
    ax = axes[i // 2, i % 2]
    ax.scatter(df[col], df['Income'], alpha=0.15, s=8, color=colors_scatter[i])
    ax.set_title(f'{col} vs Income', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Income (USD)')
    
    # Add trend line
    z = np.polyfit(df[col], df['Income'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', alpha=0.7, linewidth=2)

plt.tight_layout()
plt.suptitle('Numerical Features vs Income', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 9. Pair Plot (Key Features)

In [ ]:
# Pair plot with a subset of features (using log income for better visibility)
df_pair = df[['Age', 'Work_Experience', 'Household_Size', 'Number_of_Dependents', 'Income']].copy()
df_pair['Income_log'] = np.log1p(df_pair['Income'])

g = sns.pairplot(df_pair[['Age', 'Work_Experience', 'Household_Size', 'Income_log']],
                 diag_kind='kde', plot_kws={'alpha': 0.2, 's': 10},
                 diag_kws={'fill': True})
g.figure.suptitle('Pair Plot of Key Numerical Features', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 10. Key Findings & Summary

### Dataset Characteristics
- **10,000 records** with **13 features** (4 numerical + 9 categorical) and 1 target variable (Income)
- **No missing values** and **no duplicate** rows
- Dataset is **synthetic** with added noise

### Income Distribution
- Income is **heavily right-skewed** (skewness ≈ 2.98)
- Mean ($816K) is **~11x higher** than Median ($72K), indicating extreme outliers
- ~19% of data are outliers (IQR method)
- **Log transformation** effectively reduces skewness to ~1.35

### Feature Relationships
- **Weak linear correlations** between features and income (max |r| < 0.2)
- Homeownership status shows the highest linear correlation with income
- The weak correlations suggest **non-linear relationships** → tree-based models may perform better

### Recommendations for Modeling
1. Use **log-transformed income** as target to handle skewness
2. Try **tree-based models** (Random Forest, Gradient Boosting) which can capture non-linear patterns
3. **Feature engineering** (e.g., Age/WorkExperience ratio) may help
4. Expect **modest R² scores** due to high noise in this synthetic dataset

In [ ]:
print('=' * 60)
print('  EDA Complete!')
print('  Next step: Run model training pipeline')
print('    python main.py --train')
print('=' * 60)